# MVP Step 7 — 5-Seed Transformer Ensemble + Uncertainty-Scaled Sizing

**Setup:** Runtime → GPU (T4 or better).

**Workflow:**
1. Cell 1: install deps.
2. Cell 2: upload `algo-trading-rl.zip`.
3. Cell 3: verify GPU.
4. Cell 4: train all 5 seeds sequentially (~100k steps each). ValEvalCallback saves best-val checkpoint per seed automatically.
5. Cell 5: run `ensemble_eval.py` — reports mean + uncertainty-scaled val/test Sharpe.
6. Cell 6: package results as `colab_output.zip`. Download and drop into local workspace root.


In [ ]:
!pip install -q gymnasium stable-baselines3 yfinance ta pyyaml


In [ ]:
import os, zipfile, sys
from google.colab import files

print('Select algo-trading-rl.zip when the dialog appears...')
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
for candidate in ('algo-trading-rl', 'algo-trading-rl-main'):
    if os.path.isdir(candidate):
        os.chdir(candidate)
        break
sys.path.insert(0, os.getcwd())
print('Working dir:', os.getcwd())
assert os.path.exists('scripts/train_multi_asset_transformer.py')
assert os.path.exists('scripts/ensemble_eval.py')
assert os.path.exists('data/SPY.parquet'), 'data/*.parquet missing from zip'


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('Device:', torch.cuda.get_device_name(0))


In [ ]:
# Train 5 seeds. Each writes runs/<ts>_r007_seed{N}/ with model_best_val.zip.
# ~100k steps per seed. Serial is fine — Colab has one GPU.
import subprocess

SEEDS = [42, 101, 202, 303, 404]
for s in SEEDS:
    print(f'\n===== SEED {s} =====')
    rc = subprocess.call([
        'python', 'scripts/train_multi_asset_transformer.py',
        '--config', 'configs/transformer.yaml',
        '--tag', f'r007_seed{s}',
        '--seed', str(s),
        '--val-eval-freq', '10000',
    ])
    assert rc == 0, f'seed {s} training failed'


In [ ]:
# Collect all r007 runs into a single directory for ensemble_eval.py
import glob, shutil, os
os.makedirs('runs/ensemble_r007', exist_ok=True)
for p in glob.glob('runs/*_r007_seed*'):
    dest = os.path.join('runs/ensemble_r007', os.path.basename(p))
    if not os.path.exists(dest):
        shutil.move(p, dest)
print('Members:', sorted(os.listdir('runs/ensemble_r007')))

!python scripts/ensemble_eval.py --run-dir runs/ensemble_r007 --config configs/transformer.yaml --scale 5.0


In [ ]:
import shutil, os
shutil.make_archive('colab_output', 'zip', 'runs')
print(f'colab_output.zip ({os.path.getsize("colab_output.zip")/1024:.0f} KB)')
from google.colab import files
files.download('colab_output.zip')
